# Forensic Lab — Contract Fragmentation

Automated investigation of the most severe cases of Compra Ágil (AG) fragmentation.  
Adjust `TOP_N` and `TENDER_LIMIT` to modify the scope of the analysis.

> **Methodological note:** This analysis detects statistical indicators. It does not constitute proof of fraud. It requires independent documentary verification.

In [ ]:
import re
import pandas as pd
import os
from neo4j import GraphDatabase
from dotenv import load_dotenv
from IPython.display import display

pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.options.display.float_format = '{:,.0f}'.format

load_dotenv('../.env')
driver = GraphDatabase.driver(
    'bolt://localhost:7687',
    auth=('neo4j', os.getenv('NEO4J_ROOT_PASSWORD'))
)

def run_query(query, params=None):
    """Utility function to return Cypher results as a Pandas DataFrame."""
    with driver.session() as session:
        result = session.run(query, params or {})
        return pd.DataFrame([dict(r) for r in result])

print('Connection OK.')

## Parameters

In [ ]:
TOP_N        = 5
ORDER_LIMIT  = 6_600_000   # 100 UTM — maximum per individual AG order
TENDER_LIMIT = 66_000_000  # 1,000 UTM — Above this, a public tender was mandatory

## Fragmentation Ranking — Top N Cases

In [ ]:
df_frag = run_query("""
    MATCH (u:UnidadCompra)-[:EMITIO]->(oc:OrdenCompra_Item {Modalidad: 'AG'})-[:ADJUDICADA_A]->(p:Proveedor)
    WITH
        u.UnidadCompra AS agency,
        p.Nombre       AS vendor,
        oc.Fecha.month AS month,
        count(oc)      AS order_count,
        sum(oc.Monto)  AS total_clp
    WHERE order_count > 1 AND total_clp > $tender_threshold
    RETURN agency, vendor, month, order_count, total_clp,
           round(total_clp / $tender_threshold, 1) AS threshold_multiplier
    ORDER BY total_clp DESC
""", {'tender_threshold': TENDER_LIMIT})

print(f'{len(df_frag)} Agency-Vendor pairs above the legal threshold.')
print(f'Showing top {TOP_N}:\n')
top_frag = df_frag.head(TOP_N)
display(top_frag)

## Automated Biopsy — Case-by-Case Analysis

In [ ]:
findings = []

for _, row in top_frag.iterrows():
    agency       = row['agency']
    vendor       = row['vendor']
    month        = int(row['month'])
    order_count  = int(row['order_count'])
    total_clp    = float(row['total_clp'])
    multiplier   = round(total_clp / TENDER_LIMIT, 1)

    df_oc = run_query("""
        MATCH (u:UnidadCompra {UnidadCompra: $org})-[:EMITIO]->(oc:OrdenCompra_Item {Modalidad: 'AG'})
              -[:ADJUDICADA_A]->(p:Proveedor {Nombre: $prov})
        WHERE oc.Fecha.month = $mes
        RETURN oc.Fecha            AS date,
               oc.Monto           AS amount,
               oc.NombreEspecifico AS description,
               oc.CodigoUnico     AS id
        ORDER BY oc.Fecha ASC
    """, {'org': agency, 'prov': vendor, 'mes': month})

    if df_oc.empty:
        continue

    amounts_series = df_oc['amount'].dropna().astype(float)
    min_amount     = amounts_series.min()
    max_amount     = amounts_series.max()
    avg_amount     = amounts_series.mean()
    orders_over_limit = int((amounts_series > ORDER_LIMIT).sum())

    modal_amount = amounts_series.mode().iloc[0] if not amounts_series.mode().empty else 0
    pct_identical_amount = round(
        100 * (amounts_series == modal_amount).sum() / len(amounts_series), 1
    ) if len(amounts_series) > 0 else 0.0

    descriptions  = df_oc['description'].dropna().astype(str)
    modal_desc    = descriptions.mode().iloc[0] if not descriptions.mode().empty else ''
    pct_identical_desc = round(
        100 * (descriptions == modal_desc).sum() / len(descriptions), 1
    ) if len(descriptions) > 0 else 0.0

    ids     = df_oc['id'].dropna().astype(str)
    pattern = r'(\d+-\d+-AG25)'
    quotes  = ids.str.extract(pattern, expand=False).dropna().unique()
    base_quote    = quotes[0] if len(quotes) > 0 else ''
    unique_quotes = len(quotes)

    if pct_identical_amount > 80 and pct_identical_desc > 80:
        risk_level = 'HIGH'
    elif pct_identical_amount > 80 or pct_identical_desc > 80 or unique_quotes == 1:
        risk_level = 'MEDIUM'
    else:
        risk_level = 'LOW'

    sep = '=' * 70
    print(f'\n{sep}')
    print(f'  {agency}')
    print(f'  {vendor} | Month {month} | {order_count} orders | ${total_clp:,.0f} CLP ({multiplier}x the threshold)')
    print(sep)
    print(f'  Amount min/max/avg : ${min_amount:,.0f} / ${max_amount:,.0f} / ${avg_amount:,.0f}')
    print(f'  Orders > limit      : {orders_over_limit}')
    print(f'  % identical amount  : {pct_identical_amount}%')
    print(f'  % identical desc    : {pct_identical_desc}%')
    print(f'  Base quote          : {base_quote}')
    print(f'  Unique quotes       : {unique_quotes}')
    print(f'  SUSPICION LEVEL     : {risk_level}')
    print()

    display(df_oc[['date', 'amount', 'description', 'id']].head(10))

    findings.append({
        'agency':              agency,
        'vendor':              vendor,
        'month':               month,
        'order_count':         order_count,
        'total_clp':           total_clp,
        'multiplier':          multiplier,
        'min_amount':          min_amount,
        'max_amount':          max_amount,
        'base_quote':          base_quote,
        'unique_quotes':       unique_quotes,
        'pct_identical_amt':   pct_identical_amount,
        'pct_identical_desc':  pct_identical_desc,
        'suspicion_level':     risk_level,
        'description_sample':  modal_desc[:120],
    })

## Export Findings

In [ ]:
if findings:
    df_findings = pd.DataFrame(findings)
    output_path = 'fragmentation_findings.csv'
    df_findings.to_csv(output_path, index=False, encoding='utf-8')
    print(f'Exported: {output_path} ({len(df_findings)} cases)')
    display(df_findings)
else:
    print('No findings to export.')